# Lung Tumor Detection — Fused CT + PET with YOLO
**COMP 4500 Senior Project · Dr. Sekmen**

End-to-end pipeline on the Lung-PET-CT-Dx dataset (TCIA):

1. Environment + GPU check
2. Download expert bounding-box annotations
3. Download CT + PET scans for a few patients (Imaging Data Commons)
4. Load DICOM volumes, resample PET onto the CT grid
5. **Channel-stacking fusion**: CT lung window + CT mediastinal window + PET → one 3-channel image
6. Convert boxes to YOLO format, split train/val **by patient**
7. Train a lightweight YOLO detector (fused / CT-only / PET-only)
8. Evaluate: mAP, precision, recall, IoU
9. Streamlit demo app

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

## 1. Environment

In [ ]:
!pip -q install idc-index pydicom SimpleITK ultralytics streamlit

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Settings
`N_PATIENTS` controls how much data we pull. Each patient is roughly 0.3–1 GB. 8 patients is enough for a working demo; raise it later for real experiments.

`MODE` picks the input for the three-run comparison: `fused`, `ct`, or `pet`.

In [ ]:
N_PATIENTS = 8        # patients with both CT and PET
VAL_PATIENTS = 2      # held out for validation (split by patient, not by slice)
MODE = "fused"        # "fused" | "ct" | "pet"
EPOCHS = 40
IMG_SIZE = 512

WORK = "/content/work"
import os; os.makedirs(WORK, exist_ok=True)

## 3. Core functions
Windowing, PET normalization, fusion, and box conversion. These are the pieces to explain in the demo.

- **Lung window** (level −700 HU, width 1400): makes air-filled lung tissue and nodules visible.
- **Mediastinal window** (level 40 HU, width 350): soft tissue, vessels, lymph nodes.
- **PET**: metabolic activity (FDG uptake). Tumors light up. Clipped at the 99.5th percentile and scaled to 0–1.
- **Fusion**: stack those three as the R, G, B channels, so a standard YOLO model sees anatomy and metabolism at once with zero architecture changes.

In [ ]:
import numpy as np

def window(hu, level, width):
    lo, hi = level - width / 2, level + width / 2
    return np.clip((hu - lo) / (hi - lo), 0, 1)

def norm_pet(pet_vol):
    hi = np.percentile(pet_vol, 99.5)
    return np.clip(pet_vol / (hi + 1e-6), 0, 1)

def make_input(ct_slice, pet_slice, mode="fused"):
    lung = window(ct_slice, -700, 1400)
    medi = window(ct_slice, 40, 350)
    if mode == "fused":
        chans = [lung, medi, pet_slice]
    elif mode == "ct":
        chans = [lung, medi, lung]        # CT only, still 3 channels
    elif mode == "pet":
        chans = [pet_slice] * 3           # PET only
    else:
        raise ValueError(mode)
    return (np.stack(chans, axis=-1) * 255).astype(np.uint8)   # H x W x 3

def voc_to_yolo(xmin, ymin, xmax, ymax, w, h):
    # PASCAL VOC pixel corners -> YOLO normalized center/size
    xc = (xmin + xmax) / 2 / w
    yc = (ymin + ymax) / 2 / h
    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h
    return xc, yc, bw, bh

def iou(a, b):
    # boxes as (xmin, ymin, xmax, ymax)
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

## 4. Expert annotations
Five thoracic radiologists drew boxes in PASCAL VOC XML. Each XML file is named after the **SOPInstanceUID** of the CT slice it belongs to. Labels A/B/E/G are the cancer subtypes; for detection we treat them all as one class, `tumor`.

In [ ]:
import glob, zipfile, xml.etree.ElementTree as ET
import pandas as pd

ANN_URL = "https://www.cancerimagingarchive.net/wp-content/uploads/Lung-PET-CT-Dx-Annotations-XML-Files-rev12222020.zip"
!wget -q -nc -O {WORK}/ann.zip "{ANN_URL}"
with zipfile.ZipFile(f"{WORK}/ann.zip") as z:
    z.extractall(f"{WORK}/ann")

def parse_annotations(root_dir):
    rows = []
    for f in glob.glob(f"{root_dir}/**/*.xml", recursive=True):
        tree = ET.parse(f).getroot()
        folder = os.path.basename(os.path.dirname(f))          # e.g. A0164
        sop = os.path.splitext(os.path.basename(f))[0]
        for obj in tree.findall("object"):
            bb = obj.find("bndbox")
            rows.append(dict(
                folder=folder, sop=sop, label=obj.findtext("name"),
                xmin=float(bb.findtext("xmin")), ymin=float(bb.findtext("ymin")),
                xmax=float(bb.findtext("xmax")), ymax=float(bb.findtext("ymax")),
            ))
    return pd.DataFrame(rows)

ann = parse_annotations(f"{WORK}/ann")
ann["PatientID"] = "Lung_Dx-" + ann.folder.str.extract(r"([ABEG]\d{4})", expand=False)
print(len(ann), "boxes on", ann.sop.nunique(), "slices,", ann.PatientID.nunique(), "patients")
print(ann.label.value_counts())
ann.head()

## 5. Download CT + PET
Imaging Data Commons mirrors TCIA on public cloud storage, so no Data Retriever or login is needed. Only ~133 patients have PET, so we pick from those. For each patient we download the PET series, then CT series one at a time until we find the one containing the annotated slices.

In [ ]:
import pydicom, shutil
from idc_index import IDCClient

idc = IDCClient()
idx = idc.index[idc.index.collection_id == "lung_pet_ct_dx"]
has_pet = set(idx[idx.Modality == "PT"].PatientID)
candidates = [p for p in sorted(has_pet) if p in set(ann.PatientID)]
print(len(candidates), "patients have PET + annotations")

def series_files(d):
    return sorted(f for f in glob.glob(f"{d}/**/*", recursive=True) if os.path.isfile(f))

def download_series(uid, dest):
    idc.download_from_selection(downloadDir=dest, seriesInstanceUID=[uid],
                                dirTemplate="%SeriesInstanceUID", quiet=True)
    return f"{dest}/{uid}"

patients = {}
for pid in candidates:
    if len(patients) >= N_PATIENTS:
        break
    want = set(ann[ann.PatientID == pid].sop)
    pdir = f"{WORK}/dicom/{pid}"
    rows = idx[idx.PatientID == pid]
    pet_uid = rows[rows.Modality == "PT"].sort_values("instanceCount").SeriesInstanceUID.iloc[-1]
    ct_found = None
    for uid in rows[rows.Modality == "CT"].sort_values("series_size_MB").SeriesInstanceUID:
        d = download_series(uid, pdir)
        sops = {pydicom.dcmread(f, stop_before_pixels=True).SOPInstanceUID for f in series_files(d)}
        if sops & want:
            ct_found = d
            break
        shutil.rmtree(d)                       # not the annotated series, free the space
    if ct_found is None:
        print(pid, "- annotated CT not found, skipping"); continue
    pet_dir = download_series(pet_uid, pdir)
    patients[pid] = dict(ct=ct_found, pet=pet_dir)
    print(pid, "- CT + PET ready")

print(len(patients), "patients downloaded")

## 6. Load volumes and align PET to CT
CT is 512×512 at ~1 mm; PET is ~200×200 at ~4 mm. Both come from the same PET/CT scanner session, so they share physical coordinates. SimpleITK **resamples PET onto the CT grid** using that geometry, which gives a PET value for every CT pixel.

In [ ]:
import SimpleITK as sitk

def read_series(d):
    reader = sitk.ImageSeriesReader()
    files = reader.GetGDCMSeriesFileNames(d)
    reader.SetFileNames(files)
    return reader.Execute(), files

def load_patient(p):
    ct_img, ct_files = read_series(p["ct"])
    pet_img, _ = read_series(p["pet"])
    pet_on_ct = sitk.Resample(pet_img, ct_img, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)
    ct = sitk.GetArrayFromImage(ct_img).astype(np.float32)          # (slices, H, W) in HU
    pet = norm_pet(sitk.GetArrayFromImage(pet_on_ct))
    sop_to_z = {pydicom.dcmread(f, stop_before_pixels=True).SOPInstanceUID: i
                for i, f in enumerate(ct_files)}
    return ct, pet, sop_to_z

vols = {pid: load_patient(p) for pid, p in patients.items()}
for pid, (ct, pet, _) in vols.items():
    print(pid, "CT", ct.shape, "PET resampled", pet.shape)

## 7. Visualize the fusion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

pid = next(iter(vols))
ct, pet, sop_to_z = vols[pid]
row = ann[(ann.PatientID == pid) & ann.sop.isin(list(sop_to_z))].iloc[0]
z = sop_to_z[row.sop]
fused = make_input(ct[z], pet[z], "fused")

panels = [("CT lung window", window(ct[z], -700, 1400), "gray"),
          ("CT mediastinal window", window(ct[z], 40, 350), "gray"),
          ("PET (resampled)", pet[z], "hot"),
          ("Fused 3-channel input", fused, None)]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, img, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap); ax.set_title(title); ax.axis("off")
    ax.add_patch(patches.Rectangle((row.xmin, row.ymin), row.xmax - row.xmin, row.ymax - row.ymin,
                                   fill=False, edgecolor="lime", linewidth=2))
plt.suptitle(f"{pid}  slice {z}  expert box (green)", fontsize=14)
plt.show()

## 8. Build the YOLO dataset
Each annotated slice → one PNG + one `.txt` label (`class x_center y_center width height`, normalized 0–1).

**Split by patient**: slices from the same patient are nearly identical, so mixing them across train and val would inflate scores. Validation patients are never seen in training.

In [ ]:
from PIL import Image

def build_dataset(mode, out):
    shutil.rmtree(out, ignore_errors=True)
    pids = list(vols)
    val_set = set(pids[-VAL_PATIENTS:])
    n = {"train": 0, "val": 0}
    for pid in pids:
        ct, pet, sop_to_z = vols[pid]
        split = "val" if pid in val_set else "train"
        os.makedirs(f"{out}/images/{split}", exist_ok=True)
        os.makedirs(f"{out}/labels/{split}", exist_ok=True)
        rows = ann[(ann.PatientID == pid) & ann.sop.isin(list(sop_to_z))]
        for sop, grp in rows.groupby("sop"):
            z = sop_to_z[sop]
            img = make_input(ct[z], pet[z], mode)
            h, w = img.shape[:2]
            name = f"{pid}_{z:04d}"
            Image.fromarray(img).save(f"{out}/images/{split}/{name}.png")
            with open(f"{out}/labels/{split}/{name}.txt", "w") as fh:
                for r in grp.itertuples():
                    xc, yc, bw, bh = voc_to_yolo(r.xmin, r.ymin, r.xmax, r.ymax, w, h)
                    fh.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
            n[split] += 1
    with open(f"{out}/data.yaml", "w") as fh:
        fh.write(f"path: {out}\ntrain: images/train\nval: images/val\nnames:\n  0: tumor\n")
    print(mode, n, "val patients:", sorted(val_set))
    return f"{out}/data.yaml"

data_yaml = build_dataset(MODE, f"{WORK}/yolo_{MODE}")

## 9. Train YOLO
`yolo11n` is the nano model (~2.6M parameters): fast to train on a free Colab GPU. It starts from COCO-pretrained weights (transfer learning) and fine-tunes on our slices.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(data=data_yaml, epochs=EPOCHS, imgsz=IMG_SIZE, batch=16,
                      project=f"{WORK}/runs", name=MODE, exist_ok=True, plots=True)

## 10. Evaluate
- **Precision**: of the boxes the model drew, how many were real tumors
- **Recall**: of the real tumors, how many the model found
- **mAP@50**: average precision counting a hit when IoU ≥ 0.5
- **mAP@50-95**: stricter, averaged over IoU 0.5 → 0.95

In [ ]:
from IPython.display import Image as Show, display

run_dir = f"{WORK}/runs/{MODE}"
best = YOLO(f"{run_dir}/weights/best.pt")
m = best.val(data=data_yaml, imgsz=IMG_SIZE, plots=False, verbose=False)
print(f"{MODE}: precision={m.box.mp:.3f} recall={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")
display(Show(f"{run_dir}/results.png", width=900))

### Predictions vs. ground truth on held-out patients

In [ ]:
val_imgs = sorted(glob.glob(f"{WORK}/yolo_{MODE}/images/val/*.png"))[:4]
fig, axes = plt.subplots(1, len(val_imgs), figsize=(5 * len(val_imgs), 5))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, val_imgs):
    img = np.array(Image.open(p)); h, w = img.shape[:2]
    ax.imshow(img); ax.axis("off")
    gts = []
    for line in open(p.replace("images", "labels").replace(".png", ".txt")):
        _, xc, yc, bw, bh = map(float, line.split())
        g = ((xc - bw/2) * w, (yc - bh/2) * h, (xc + bw/2) * w, (yc + bh/2) * h); gts.append(g)
        ax.add_patch(patches.Rectangle(g[:2], g[2]-g[0], g[3]-g[1], fill=False, ec="lime", lw=2))
    pred = best.predict(p, verbose=False)[0].boxes
    for b, s in zip(pred.xyxy.cpu().numpy(), pred.conf.cpu().numpy()):
        best_iou = max((iou(b, g) for g in gts), default=0)
        ax.add_patch(patches.Rectangle(b[:2], b[2]-b[0], b[3]-b[1], fill=False, ec="red", lw=2))
        ax.text(b[0], b[1] - 4, f"{s:.2f} IoU {best_iou:.2f}", color="red", fontsize=9)
    ax.set_title(os.path.basename(p))
plt.suptitle("green = expert box, red = model prediction"); plt.show()

## 11. Three-run comparison (optional, ~3× training time)
Same patients, same split, only the input changes. This is the core experiment of the project.

In [ ]:
RUN_COMPARISON = False   # set True when you have time

if RUN_COMPARISON:
    scores = {}
    for mode in ["ct", "pet", "fused"]:
        y = build_dataset(mode, f"{WORK}/yolo_{mode}")
        YOLO("yolo11n.pt").train(data=y, epochs=EPOCHS, imgsz=IMG_SIZE, batch=16,
                                  project=f"{WORK}/runs", name=mode, exist_ok=True, verbose=False)
        r = YOLO(f"{WORK}/runs/{mode}/weights/best.pt").val(data=y, imgsz=IMG_SIZE, verbose=False)
        scores[mode] = dict(precision=r.box.mp, recall=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map)
    print(pd.DataFrame(scores).T.round(3))

## 12. Save to Google Drive
Colab wipes `/content` when the session ends. This keeps the trained weights and plots.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/lung_tumor_capstone"
os.makedirs(dest, exist_ok=True)
shutil.copytree(f"{WORK}/runs", f"{dest}/runs", dirs_exist_ok=True)
shutil.copy(f"{WORK}/runs/{MODE}/weights/best.pt", f"{dest}/best.pt")
print("Saved to", dest)

## 13. Streamlit demo app
Upload a fused slice PNG (from `yolo_fused/images/val`) and the model draws its detections. Run the next cell, then open the `trycloudflare.com` link it prints.

In [ ]:
app = r"""
import streamlit as st
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

st.set_page_config(page_title="Lung Tumor Detector", layout="centered")
st.title("Lung Tumor Detection: Fused CT + PET")
st.caption("YOLO11n trained on Lung-PET-CT-Dx. Channels: CT lung window, CT mediastinal window, PET.")

@st.cache_resource
def load():
    return YOLO("WEIGHTS")

model = load()
conf = st.slider("Confidence threshold", 0.05, 0.9, 0.25, 0.05)
f = st.file_uploader("Upload a fused slice (PNG)", type=["png", "jpg"])
if f:
    img = Image.open(f).convert("RGB")
    res = model.predict(np.array(img), conf=conf, verbose=False)[0]
    draw = ImageDraw.Draw(img)
    for (x1, y1, x2, y2), s in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)
        draw.text((x1, max(0, y1 - 12)), f"tumor {s:.2f}", fill=(255, 0, 0))
    st.image(img, caption=f"{len(res.boxes)} detection(s)")
    c1, c2, c3 = st.columns(3)
    arr = np.array(Image.open(f).convert("RGB"))
    c1.image(arr[..., 0], caption="CT lung", clamp=True)
    c2.image(arr[..., 1], caption="CT mediastinal", clamp=True)
    c3.image(arr[..., 2], caption="PET", clamp=True)
"""
with open("/content/app.py", "w") as fh:
    fh.write(app.replace("WEIGHTS", f"{WORK}/runs/{MODE}/weights/best.pt"))

!wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
!nohup streamlit run /content/app.py --server.port 8501 --server.headless true > /content/st.log 2>&1 &
!nohup /content/cloudflared tunnel --url http://localhost:8501 > /content/cf.log 2>&1 &
import time, re; time.sleep(12)
m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
print("Open the app:", m.group(0) if m else "not ready yet, rerun: !grep trycloudflare /content/cf.log")

## Tools used
| Tool | What it does here |
|---|---|
| Google Colab + T4 GPU | Free GPU environment |
| PyTorch | Deep learning framework under YOLO |
| Ultralytics YOLO11 | Object detector (training, validation, inference) |
| idc-index | Downloads TCIA data from Imaging Data Commons |
| pydicom | Reads DICOM headers (SOPInstanceUID mapping) |
| SimpleITK | Loads 3D volumes, resamples PET onto the CT grid |
| NumPy | Windowing, normalization, channel stacking |
| Matplotlib | Visualization |
| Pandas | Annotation table |
| Streamlit + cloudflared | Web demo app |

**Dataset:** Li, P. et al. (2020). A Large-Scale CT and PET/CT Dataset for Lung Cancer Diagnosis (Lung-PET-CT-Dx). The Cancer Imaging Archive. https://doi.org/10.7937/TCIA.2020.NNC2-0461